# 第8周 · 周六：AB实验全流程模拟——从设计到决策的完整闭环

> 难度：⭐⭐⭐⭐ | 定位：完整模拟一次AB实验的8个步骤，将W7-W8所有知识整合为实践能力
> 前置依赖：W7-W8所有实验知识 | 后续用途：独立主导一个AB实验全流程
> 面试地位："请描述你最近做的一个AB实验的全过程"——这是最常被问到的综合题
> 配对课程：01python_learn W8D6（中期大项目 上）

---

## 📘 区块1：核心概念与第一性原理

### 1.1 一次完整AB实验的8个步骤

```
步骤1：需求澄清（OSM）      → 为什么做这个实验？
步骤2：实验设计             → 怎么设计？（指标/单位/分流/周期）
步骤3：样本量/MDE计算       → 跑多少用户？跑多久？
步骤4：实施与监控           → 实验上线，监控SRM和数据质量
步骤5：数据分析             → 实验结束，分析结果
步骤6：决策                 → 上线/放弃/延长？
步骤7：报告与归档           → 记录实验结论，沉淀为组织知识
步骤8：回顾与迭代           → 3个月后回看，验证长期效果
```

这8个步骤不是线性的——步骤5发现数据有问题可能要回到步骤2重新设计，步骤6决定延长实验可能要回到步骤4继续跑。但每一步都是完整实验不可或缺的。

### 1.2 分析师在实验中的三种角色

- **设计师**（步骤1-3）：定义问题、设计实验、计算样本量——这是分析师最有价值的角色
- **裁判员**（步骤4-6）：监控数据质量、分析结果、做出决策——这是分析师最核心的角色
- **历史学家**（步骤7-8）：记录实验、回顾验证、沉淀知识——这是分析师最被忽视但最重要的角色

### 1.3 本次模拟的场景设定

你是某电商平台的首席数据分析师。产品团队提出了一个新推荐算法（协同过滤替代热度排序），期望提升用户人均下单量。你需要主导整个AB实验，从设计到最终决策。

**关键参数**：
- 当前人均月下单量：1.5单/月
- 产品期望提升：+10%（从1.5→1.65单/月）
- 人均下单量标准差：2.0单
- 平台每日活跃用户：50000人
- 实验分流比例：50:50
- 主要指标：人均下单量
- 护栏指标：客单价（不能降）、退款率（不能升）
- 显著性水平：α=0.05, power=0.80

### 1.4 5个必须的检查点

每次实验决策前，必须通过这5个检查点：
1. **SRM检查**：两组人数比例是否偏离50:50？
2. **主指标检查**：p值如何？效应量如何？CI如何？
3. **效应量检查**：效应量是否 > MDE？是否有业务意义？
4. **护栏指标检查**：护栏指标是否显著恶化？
5. **Peeking检查**：中间是否偷看过数据？如果有，是否用了序贯检验？

只有5个检查点全部通过，才能做出"建议上线"的决策。

### 1.5 面试高频考点

**面试官常问**：
- "请完整描述你做过的一次AB实验的全流程"
- "实验不显著怎么办？"
- "你怎么判断一个实验是否可以上线？"
- "你在这个实验中扮演什么角色？"

**回答框架**：用8步骤框架组织回答——不要跳步，不要略过"为什么这么设计"。面试官想听的不只是"结果是什么"，更是"你怎么思考的"。</cell>


In [ ]:
## 🧩 区块2：知识点详细拆解——8步骤深度解析

### 2.1 步骤1：需求澄清——用OSM框架

**OSM = Objective（目标）+ Strategy（策略）+ Measurement（度量）**

| 要素 | 问题 | 本次实验 |
|------|------|---------|
| O | 我们想解决什么问题？ | 用户人均下单量偏低（1.5单/月） |
| S | 用什么策略解决？ | 用协同过滤替代热度排序 |
| M | 怎么知道策略有效？ | 主指标：人均下单量；护栏：客单价、退款率 |

**常见错误**：跳过OSM直接设计实验→不知道"成功"长什么样→实验结束后不知道怎么做决策。

### 2.2 步骤2：实验设计——8个设计决策

1. **实验单元**：user_id（用户级随机化）
2. **分流比例**：50:50（最大化统计功效）
3. **层（Layer）**：算法层（不影响UI层和内容层实验）
4. **实验周期**：2周（14天，覆盖1-2个完整用户行为周期）
5. **主要指标**：人均下单量（连续变量，用均值类指标方法）
6. **护栏指标**：客单价（不能降）、退款率（不能升）
7. **分组方式**：hash(user_id + layer_seed + experiment_id) % 2
8. **排除规则**：新用户（注册<7天）、测试账号、内部员工

### 2.3 步骤3：样本量/MDE——量化决策边界

**均值类指标样本量公式**：
```
n_per_group = 2(Zα/2 + Zβ)² × σ² / δ²
```

代入本题参数：
- σ = 2.0
- δ = 1.5 × 0.10 = 0.15（相对10%提升对应的绝对提升）
- Zα/2 = 1.96, Zβ = 0.84
- n = 2 × (1.96 + 0.84)² × 4 / 0.0225 ≈ 2 × 7.84 × 4 / 0.0225 ≈ 2788 → 取3000

**反向MDE**：如果只能跑2周（每组约350000人），反向MDE ≈ 0.05单（相对3.3%）。我们能检测到比产品期望（10%）更小的效果——样本量充足。

### 2.4 步骤4：实施与监控——"跑实验不是等实验"

实验上线后，你不是"等2周后看结果"，而是每天监控：
- SRM是否正常（每天检查）
- 数据是否有异常（突然的0值、极端值）
- 是否有crash率异常（实验组bug）
- 是否有明显的负面效果（护栏指标）

### 2.5 步骤5：数据分析——不只是看p值

完整的分析包括：
1. SRM检验（第一道关）
2. 主指标t检验（CUPED调整前后对比）
3. 效应量和95% CI
4. 护栏指标t检验
5. 分层分析（iOS vs Android、新用户 vs 老用户）
6. 时间趋势分析（效果是否稳定？新奇效应是否消退？）

### 2.6 步骤6：决策——"上线"不是唯一答案

| 情况 | 主指标 | 护栏 | 效应量 | 决策 |
|------|--------|------|--------|------|
| A | p<0.05 | ✓无恶化 | >MDE且有业务意义 | ✅ 建议上线 |
| B | p<0.05 | ✗有恶化 | — | ❌ 不上线（护栏恶化） |
| C | p>0.05 | ✓无恶化 | n不够→延长实验 | ⏸ 延长实验 |
| D | p>0.05 | ✓无恶化 | n够了→真没效果 | ❌ 放弃策略 |
| E | p<0.05 | ✓无恶化 | <MDE→无业务意义 | ⚠️ 显著但无意义 |

### 2.7 步骤7：报告与归档——"做完实验不等于结束"

一份完整的实验报告包括：背景、设计、样本量、结果（主指标+护栏+分层）、决策、风险提示、后续建议。

**归档的价值**：3个月后有新人问"推荐算法之前有没有人测过？"——不需要重新做一遍。

### 2.8 步骤8：回顾与迭代——3个月后的回看

实验上线3个月后，回顾：
- 全量效果和实验效果一致吗？（如果差异大→为什么？新奇效应？季节变化？）
- 护栏指标有没有出现实验中没发现的问题？
- 用户的长期行为是否验证了实验结论？</cell>


## 📝 区块7：当日小结

> **今天最重要的一个认知**：一次完整的AB实验 = 8个步骤 + 5个检查点。不是"p<0.05就上线"，而是"SRM通过 + 主指标显著 + 效应量有业务意义 + 护栏无恶化 + 无Peeking = 上线"。实验决策不是二元的"显著/不显著"，而是五元的（A/B/C/D/E五种情况）。分析师的核心价值在于设计实验和做出判断，而不只是"跑t检验"。

**五条金句带回家**：
1. 8步骤：需求→设计→样本量→监控→分析→决策→报告→回顾
2. 5检查点：SRM + 主指标 + 效应量 + 护栏 + Peeking
3. "不显著"有4种可能：样本不够、效果太小、策略没效、数据有问题
4. 护栏指标是"刹车"——GMV涨了但退款率飙升=这个策略不能上
5. 一份完整的实验报告比一个"显著的p值"更有长远价值

---

*03analyst_thinking · Week 8 Day 6 · AB实验全流程模拟 · 2026-05-21*</cell>


In [ ]:
import numpy as np
from scipy import stats
np.random.seed(42)

print("=" * 60)
print("步骤1：需求澄清 → OSM")
print("=" * 60)
print("  O: 提升用户人均下单量（从月均1.5单→1.65单，+10%）")
print("  S: 用协同过滤替代热度排序")
print("  M: 主指标=人均下单数，护栏=客单价、退款率")

print("\n" + "=" * 60)
print("步骤2：设计实验")
print("=" * 60)
print("  随机化单元: user_id")
print("  分流比例: 50:50")
print("  实验周期: 2周（14天）")
print("  排除: 新用户(<7天)、测试账号、内部员工")

print("\n" + "=" * 60)
print("步骤3：计算样本量")
print("=" * 60)
baseline, mde_rel, std_orders = 1.5, 0.10, 2.0

def calc_n_mean(std, mde, alpha=0.05, power=0.80):
    za = stats.norm.ppf(1 - alpha / 2)
    zb = stats.norm.ppf(power)
    return int(np.ceil(2 * (za + zb)**2 * std**2 / mde**2))

mde_abs = baseline * mde_rel
n_needed = calc_n_mean(std=std_orders, mde=mde_abs)
daily_users = 50000
n_available = daily_users * 14 // 2
mde_achievable = np.sqrt(2) * (stats.norm.ppf(0.975) + stats.norm.ppf(0.80)) * std_orders / np.sqrt(n_available)

print(f"  人均下单数std≈{std_orders}, MDE={mde_abs:.2f}单→每组需{n_needed:,}人")
print(f"  2周可获每组{n_available:,}人→可检测MDE={mde_achievable:.2f}单(相对{mde_achievable/baseline*100:.1f}%)")
print(f"  ✓ 样本量充足，可以跑实验")

print("\n" + "=" * 60)
print("步骤4：实施与监控——模拟14天数据")
print("=" * 60)

n = n_available
c_orders = np.random.poisson(baseline, n)
e_orders = np.random.poisson(baseline * (1 + mde_rel), n)
c_gmv = np.random.lognormal(3.5, 0.6, n) * c_orders
e_gmv = np.random.lognormal(3.5, 0.6, n) * e_orders
c_refund = np.random.binomial(1, 0.05, n)
e_refund = np.random.binomial(1, 0.05, n)

print("  实验已上线，每日监控中...")
print("  第3天SRM检查: ✓ 50:50")
print("  第7天数据检查: ✓ 数据正常")
print("  第14天实验结束")

print("\n" + "=" * 60)
print("步骤5：分析结果")
print("=" * 60)

# SRM
print(f"\n  [检查1] SRM: 对照{n}人 vs 实验{n}人 → 50:50 ✓")

# 主指标
t1, p1 = stats.ttest_ind(c_orders, e_orders)
effect = e_orders.mean() - c_orders.mean()
effect_rel = (e_orders.mean() / c_orders.mean() - 1) * 100
se_pooled = np.sqrt(c_orders.var()/n + e_orders.var()/n)
ci = stats.t.interval(0.95, 2*n-2, loc=effect, scale=se_pooled)
cohens_d = effect / np.sqrt((c_orders.var() + e_orders.var()) / 2)

print(f"\n  [检查2] 主指标(人均下单):")
print(f"    对照{c_orders.mean():.2f}, 实验{e_orders.mean():.2f}")
print(f"    提升{effect_rel:.1f}%, p={p1:.4f}, 95%CI=[{ci[0]:.2f},{ci[1]:.2f}]")
print(f"    Cohen's d={cohens_d:.3f} {'✓ 显著' if p1 < 0.05 else '✗ 不显著'}")

# 效应量
print(f"\n  [检查3] 效应量: {effect:.2f}单")
print(f"    MDE_design={mde_abs:.2f}单, 实际效果={effect:.2f}单")
print(f"    {'✓ 效应量>MDE且有业务意义' if effect > mde_abs else '⚠ 效应量不足'}")

# 护栏
t2, p2 = stats.ttest_ind(c_gmv, e_gmv)
t3, p3 = stats.ttest_ind(c_refund.astype(float), e_refund.astype(float))
print(f"\n  [检查4] 护栏指标:")
print(f"    人均GMV: p={p2:.4f} {'✓ 无显著差异' if p2 > 0.05 else '⚠ 有差异！'}")
print(f"    退款率: p={p3:.4f} {'✓ 无显著差异' if p3 > 0.05 else '⚠ 有差异！'}")

# CUPED分析（延迟1天导入实验前数据）
print(f"\n  [CUPED分析] 假设有实验前30天数据（ρ≈0.7）:")
X = baseline + np.random.normal(0, 0.5, n)  # 模拟实验前数据
theta = np.cov(c_orders, X)[0, 1] / np.var(X)
X_mean = X.mean()
c_adj = c_orders - theta * (X - X_mean)
e_adj = e_orders - theta * (X - X_mean)
var_red = (1 - np.var(c_adj) / np.var(c_orders)) * 100
t_cuped, p_cuped = stats.ttest_ind(c_adj, e_adj)
print(f"    方差缩减={var_red:.1f}%, CUPED p={p_cuped:.4f} vs 传统p={p1:.4f}")
print(f"    → CUPED让检验更敏感")

print("\n" + "=" * 60)
print("步骤6：决策")
print("=" * 60)

if p1 < 0.05 and ci[0] > 0 and effect > mde_abs and p2 > 0.05 and p3 > 0.05:
    print(f"  ✅ 建议上线新算法")
    print(f"  预期每用户多下单{ci[0]:.2f}-{ci[1]:.2f}次")
elif p1 < 0.05 and (p2 < 0.05 or p3 < 0.05):
    print(f"  ❌ 不上线——护栏指标恶化")
else:
    print(f"  ⏸ 证据不足，建议延长实验或放弃")

print("\n" + "=" * 60)
print("步骤7：实验报告一句话总结")
print("=" * 60)
print(f"  '新推荐算法使人均下单量从{c_orders.mean():.2f}提升到{e_orders.mean():.2f}")
print(f"   （+{effect_rel:.1f}%，p={p1:.4f}, 95%CI=[{ci[0]:.2f},{ci[1]:.2f}], Cohen's d={cohens_d:.3f}），")
print(f"   护栏指标（客单价、退款率）无显著恶化，建议全量上线。'")
print(f"  → 归档至实验知识库 #EXP-2024-001")

print("\n" + "=" * 60)
print("步骤8：回顾与迭代（3个月后回看）")
print("=" * 60)
print("  全量后3个月实际提升: +9.5%（接近实验测到的10%左右）")
print("  长期留存: 无显著变化")
print("  结论: 实验结论被长期数据验证 ✓")

## 📊 区块4：业务场景应用

### 场景1：实验不显著——诊断与沟通

如果这个实验结果p=0.25（不显著），你应该怎么做？

1. 检查SRM ✓
2. 反向计算MDE：每组350000人→MDE=0.05单→产品期望0.15单远大于MDE→样本量够了
3. 结论：样本量够了但不显著→策略真的没效果（或效果<0.05单）
4. 沟通："实验跑了2周，每组35万人，样本量足够检测到0.05单的差异。但结果显示两组几乎没有差异。这意味着推荐算法的改进没有带来下单量的提升——不是因为样本不够，而是策略本身效果太小。"

### 场景2：护栏指标告警——如何说"不"

如果GMV显著提升但退款率也从5%升到7%（p=0.01）：

1. 分析退款率上升的原因：新推荐算法推荐的更多是"低价高退款率"商品
2. 计算净效果：GMV涨了5%但利润率降了3%→净效果可能是负的
3. 沟通："新算法的确提升了GMV（+5%），但它推荐的更多低质商品导致退款率从5%升到7%。如果我们只看GMV，会觉得算法很好。但算上退款后的净GMV实际下降了。建议优化算法——在推荐中加入'商品质量分'来过滤高退款率商品。"

### 场景3：小公司版实验——流量不够怎么办

如果你的平台每天只有1000活跃用户（不是50000），同样测这个推荐算法：

- 2周每组只有7000人
- 反向MDE=0.36单（相对24%）
- 产品期望+10%（0.15单）远小于MDE→2周检测不到

**怎么办**：
1. 延长实验到8周（每组28000人→MDE=0.18单，还是不够）
2. 接受"只能检测大效果"的现实→如果推荐算法真有超过24%的提升才值得测
3. 做大改动而非小优化→用更大的策略差异来产生更大的效应</cell>


## ✅ 区块5：分级检验清单

### 🟢 初级检验
- [ ] 能说出AB实验的8个步骤
- [ ] 能独立完成样本量计算（均值类和比例类）
- [ ] 能在看实验结果前检查SRM
- [ ] 能区分5种决策情况（A/B/C/D/E）

### 🟡 中级检验
- [ ] 能在实验结果中同时呈现p值、效应量、CI
- [ ] 能处理"不显著"的诊断（样本不够 vs 没效果）
- [ ] 能分析护栏指标的结果
- [ ] 能写出一份完整的实验报告

### 🔴 高级检验
- [ ] 能从OSM开始独立主导一个完整AB实验
- [ ] 能在实验中融入CUPED、分层分析
- [ ] 能向不同受众（产品/运营/工程）汇报实验结论
- [ ] 能设计多个实验之间的优先级和资源分配方案</cell>


## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **←** W7假设检验 | t检验、p值、CI | 数据分析的核心工具 |
| **←** W8D1 MDE | 样本量计算、反向MDE | 步骤3的核心计算 |
| **←** W8D2 CUPED | 方差缩减 | 步骤5的进阶分析 |
| **←** W8D5 避坑大全 | SRM/Peeking/护栏 | 步骤4-6的检查清单 |
| **→** W9因果推断 | DID/PSM | 当AB实验不可行时的替代方案 |
| **→** 日常工作 | 独立主导实验 | 本课是W7-W8的整合应用 |</cell>


---

## 二、当日小结

> **今天最重要的一个认知**：一次完整的AB实验 = 8步 + 5个检查点（SRM/主指标/效应量/护栏/Peeking）。不是"p<0.05就上线"，而是"多方面证据一致指向同一个结论时才上线"。

---

*03analyst_thinking · Week 8 Day 6 · 2026-05-21*